# Corbel — YOLOv8n floor plan detection training

Trains the wall/door/window detector used by Corbel's browser-side plan import
(`corbel-detect.onnx`). Classes: 0=wall, 1=door, 2=window. Dataset: CubiCasa5k
converted to YOLO boxes (4200 train / 400 val / 400 test images).

## Before you run this

1. **Set the GPU runtime**: Runtime → Change runtime type → T4 GPU (free tier).
2. **Upload the dataset to your Google Drive root** (`MyDrive`, not a
   subfolder): `ml/corbel-dataset.zip` (1.8 GB) from this repo. Do this once —
   Drive persists it across sessions, so you won't need to re-upload for a
   resumed or repeat run.
3. Run the cells in order (Runtime → Run all works too).

Training runs ~150 epochs with early stopping (`patience=30`, so it may finish
sooner if validation loss plateaus). On a free T4 this is roughly **6–12
hours** — Colab free-tier sessions can disconnect before that; checkpoints
save to Drive every epoch (cell 5), so if it disconnects, re-open this
notebook, set `RESUME = True` in cell 5, and run cells 1–2 then 5 onward again.

Epoch count was raised from an original 50 after a 2026-09-05 benchmark showed
the previously-shipped model's boxes were imprecise in a way no amount of
confidence/NMS threshold tuning or box-merging post-processing could fix —
pointing at undertraining rather than a labeling or metric bug (see
`corbel/benchmark_out/comparison-40/` and `corbel/context.md` in the repo for
the full writeup). This is a reasoned starting point, not a verified
hyperparameter search.</cell id="987e651b">


In [ ]:
# 1 — GPU check (should show a Tesla T4; if this errors, set Runtime > Change
# runtime type > T4 GPU first, then Runtime > Restart session)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2 — mount Drive, unpack dataset
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/corbel-dataset.zip -d /content/

In [ ]:
# 3 — install ultralytics
!pip install -q ultralytics

In [ ]:
# 4 — point dataset yaml at Colab path (yaml carries an absolute local-machine path)
!sed -i 's|^path:.*|path: /content/dataset|' /content/dataset/cubicasa.yaml
!cat /content/dataset/cubicasa.yaml

In [ ]:
# 5 — train (checkpoints to Drive every epoch; survives disconnects)
# epochs raised 50->150 with patience=30 early stopping: a 2026-09-05 benchmark
# showed the shipped model's boxes are imprecise in a way threshold tuning and
# post-processing can't fix, pointing at undertraining rather than a labeling
# bug (see corbel/benchmark_out/comparison-40/). Not verified by an actual
# training run here (no GPU in that session) - patience is the safety net if
# 150 is more than needed.
from ultralytics import YOLO

RESUME = False  # set True to continue an interrupted run

if RESUME:
    model = YOLO('/content/drive/MyDrive/corbel-runs/corbel/weights/last.pt')
    model.train(resume=True)
else:
    model = YOLO('yolov8n.pt')
    model.train(
        data='/content/dataset/cubicasa.yaml',
        epochs=150,
        patience=30,
        imgsz=640,
        save_period=1,
        project='/content/drive/MyDrive/corbel-runs',
        name='corbel',
        exist_ok=True,
    )

In [ ]:
# 6 — evaluate on held-out test split (mAP / precision / recall per class)
from ultralytics import YOLO

best = YOLO('/content/drive/MyDrive/corbel-runs/corbel/weights/best.pt')
metrics = best.val(data='/content/dataset/cubicasa.yaml', split='test')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('per-class mAP50:', dict(zip(['wall', 'door', 'window'], metrics.box.maps.tolist())))

In [ ]:
# 7 — export ONNX for browser inference (onnxruntime-web)
onnx_path = best.export(format='onnx', imgsz=640, opset=17)
!cp {onnx_path} /content/drive/MyDrive/corbel-runs/corbel-detect.onnx
print('Saved to Drive: corbel-runs/corbel-detect.onnx')
print('Then locally:  cp corbel-detect.onnx corbel/public/models/corbel-detect.onnx')

## Bring the model back to Corbel

1. Download `corbel-detect.onnx` from your Google Drive: `corbel-runs/corbel-detect.onnx`.
2. Locally: `cp ~/Downloads/corbel-detect.onnx corbel/public/models/corbel-detect.onnx`
3. Re-run the benchmark against the **test** split (not val — val was used for
   all the threshold/post-processing tuning that preceded this training run,
   so val numbers here would be optimistic):
   ```
   cd corbel
   node --experimental-strip-types scripts/benchmark-local-yolo.ts --split test --n 400
   ```
4. Compare honestly against the pre-retrain baseline (bbox-IoU@0.50 F1):
   wall 76.7%, door 86.5%, window 87.4%. Door/window should be much closer to
   95% now; wall may still fall short — see `corbel/context.md` for why that's
   expected rather than a sign something went wrong.